
# Kuroshio Daily 1993–2025 — 人工复核 / 标注工具
## 只处理自动流水线标记的可疑日期

这个 Notebook 继承之前 MANUAL22 的操作逻辑，但针对 1993–2025 长序列做了以下改进：

- 自动读取 `07_人工复核队列.csv`；
- 每次只保持一张交互图，避免 WSL 绘图内存峰值；
- 显示当天 50 m speed field；
- 显示自动路径；
- 显示前/后已自动接受路径；
- 2023–2025 如果旧 `R1_FINAL_MANUAL22` 中存在历史人工结果，会额外以紫色虚线作为**参考**；
- 增加 **“自动路径正确，直接批准”** 按钮；
- 真正错误时再鼠标标控制点并做局地速度脊细化；
- 支持“该日不可用”；
- 全部人工队列处理完成后自动冻结最终 tracks，并生成长时段 dual-branch dataset。

### 鼠标

- 左键：添加控制点；
- 右键：删除距离鼠标最近的控制点。

### 推荐操作

1. 先看自动路径；
2. 自动路径其实正确 → 点“批准自动路径”；
3. 自动路径错误 → “邻日初始化”或直接左键画控制点；
4. “构建/细化”；
5. 勾选确认并保存人工路径；
6. 数据本身无法判断 → 标记“不可用”。


In [ ]:

# ===== 0. 配置 =====
from pathlib import Path
from dataclasses import dataclass
import os,json,math,shutil,gc,warnings
import numpy as np
import pandas as pd

@dataclass
class CFG:
    PROJECT_ROOT: str="/mnt/d/KuroshioDaily1993_2025"

    LEGACY_REFERENCE_TRACK_DIR: str=(
        "/mnt/d/KuroshioDailyR1/"
        "R1_FINAL_MANUAL22/tracks_final"
    )

    XMIN: float=120.0
    XMAX: float=148.0
    YMIN: float=20.0
    YMAX: float=42.0

    DISPLAY_STRIDE: int=2
    QUIVER_STRIDE: int=18

    CONTROL_SNAP_RADIUS_KM: float=18.0
    CONTROL_SNAP_DISTANCE_PENALTY: float=0.35

    PATH_SAMPLE_SPACING_KM: float=10.0
    RIDGE_HALF_KM: float=24.0
    RIDGE_SAMPLES: int=13
    RIDGE_DISTANCE_PENALTY: float=0.22
    RIDGE_REFINE_PASSES: int=2

    MIN_CONTROL_POINTS: int=8
    MIN_MAX_LON: float=145.8
    MIN_FINITE_SPEED_FRAC: float=0.95

    WARN_RIDGE_MEDIAN: float=0.85
    WARN_RIDGE_P10: float=0.60
    WARN_P95_TURN_DEG: float=145.0

    TEMPORAL_INIT_CONTROL_POINTS: int=22

cfg=CFG()

ROOT=Path(cfg.PROJECT_ROOT)

REPORTS=ROOT/"reports"
AUTO_TRACK=ROOT/"tracks_auto"
MANUAL_CACHE=ROOT/"manual_cache_50m"

MANUAL_ROOT=ROOT/"manual_qc"
MANUAL_TRACK=MANUAL_ROOT/"tracks_manual"
MANUAL_META=MANUAL_ROOT/"metadata"
MANUAL_FIG=MANUAL_ROOT/"figures"
MANUAL_REPORT=MANUAL_ROOT/"reports"

FINAL_ROOT=ROOT/"final"
FINAL_TRACK=FINAL_ROOT/"tracks_final"
FINAL_DATASET=FINAL_ROOT/"dataset"
FINAL_REPORT=FINAL_ROOT/"reports"

LEGACY_REF=Path(
    cfg.LEGACY_REFERENCE_TRACK_DIR
)

for p in [
    MANUAL_ROOT,MANUAL_TRACK,
    MANUAL_META,MANUAL_FIG,
    MANUAL_REPORT,
    FINAL_ROOT,FINAL_TRACK,
    FINAL_DATASET,FINAL_REPORT
]:
    p.mkdir(parents=True,exist_ok=True)

queue_path=REPORTS/"07_人工复核队列.csv"
auto_report_path=REPORTS/"05_自动追踪结果.csv"
manifest_path=REPORTS/"01_最终源合并Manifest.csv"

for p in [
    queue_path,
    auto_report_path,
    manifest_path
]:
    if not p.exists():
        raise FileNotFoundError(p)

queue=pd.read_csv(
    queue_path,
    parse_dates=["date"]
).sort_values("date")

auto_report=pd.read_csv(
    auto_report_path,
    parse_dates=["date"]
).sort_values("date")

manifest=pd.read_csv(
    manifest_path,
    parse_dates=["date"]
).sort_values("date")

DATES=[
    pd.Timestamp(x).normalize()
    for x in queue["date"]
]

print("人工复核队列:",len(DATES))


In [ ]:

# ===== 1. 交互环境 =====
import importlib.util

missing=[
    x for x in ["ipympl","ipywidgets","scipy"]
    if importlib.util.find_spec(x) is None
]
if missing:
    raise RuntimeError(
        "缺少: "+", ".join(missing)+"\n"
        "WSL终端执行:\n"
        "python -m pip install -U ipympl ipywidgets scipy"
    )

get_ipython().run_line_magic(
    "matplotlib","widget"
)

import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.interpolate import (
    RegularGridInterpolator,
    PchipInterpolator,
    interp1d
)

plt.ioff()
print("backend:",plt.get_backend())


In [ ]:

# ===== 2. 几何 / 读轨迹 / 人工细化 =====
EARTH_R_KM=6371.0088

def wrap_pi(a):
    return (a+np.pi)%(2*np.pi)-np.pi

def haversine_km(lon1,lat1,lon2,lat2):
    lon1,lat1,lon2,lat2=map(np.asarray,(lon1,lat1,lon2,lat2))
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    dp=p2-p1
    aa=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*EARTH_R_KM*np.arcsin(np.minimum(1.0,np.sqrt(aa)))

def angle_east_rad(lon1,lat1,lon2,lat2):
    p1,p2=np.deg2rad(lat1),np.deg2rad(lat2)
    dl=np.deg2rad(lon2-lon1)
    y=np.sin(dl)*np.cos(p2)
    x=np.cos(p1)*np.sin(p2)-np.sin(p1)*np.cos(p2)*np.cos(dl)
    nav=np.arctan2(y,x)
    return wrap_pi(np.pi/2-nav)

def forward_point(lon,lat,east_angle,distance_km):
    nav=np.pi/2-east_angle
    phi1,lam1=np.deg2rad(lat),np.deg2rad(lon)
    d=distance_km/EARTH_R_KM
    phi2=np.arcsin(
        np.sin(phi1)*np.cos(d)+
        np.cos(phi1)*np.sin(d)*np.cos(nav)
    )
    lam2=lam1+np.arctan2(
        np.sin(nav)*np.sin(d)*np.cos(phi1),
        np.cos(d)-np.sin(phi1)*np.sin(phi2)
    )
    return float(np.rad2deg(lam2)),float(np.rad2deg(phi2))

def load_field(dt):
    p=MANUAL_CACHE/f"{pd.Timestamp(dt):%Y%m%d}.npz"
    if not p.exists():
        raise FileNotFoundError(
            f"缺少人工缓存 {p}。"
            "回到DataBuild Notebook重新运行人工队列缓存单元。"
        )
    with np.load(p) as z:
        return {
            "lon":np.asarray(z["lon"],float),
            "lat":np.asarray(z["lat"],float),
            "u":np.asarray(z["u"],np.float32),
            "v":np.asarray(z["v"],np.float32),
        }

def load_track(path):
    p=Path(path)
    if not p.exists():
        return None
    with np.load(p,allow_pickle=False) as z:
        return {
            "lon":np.asarray(z["lon"],float),
            "lat":np.asarray(z["lat"],float),
            "completed":(
                bool(z["completed"])
                if "completed" in z.files
                else True
            ),
        }

def auto_track(dt):
    return load_track(
        AUTO_TRACK/f"{pd.Timestamp(dt):%Y%m%d}.npz"
    )

def legacy_track(dt):
    p=LEGACY_REF/f"{pd.Timestamp(dt):%Y%m%d}.npz"
    return load_track(p) if p.exists() else None

def accepted_neighbor(dt,direction):
    dt=pd.Timestamp(dt)
    q=auto_report[
        auto_report["status"]=="accepted_auto"
    ].copy()

    if direction=="prev":
        q=q[q["date"]<dt].sort_values("date")
        if not len(q):
            return None,None
        d=pd.Timestamp(q.iloc[-1].date)
    else:
        q=q[q["date"]>dt].sort_values("date")
        if not len(q):
            return None,None
        d=pd.Timestamp(q.iloc[0].date)

    return d,auto_track(d)

def resample_polyline(lon,lat,n=220):
    lon=np.asarray(lon,float)
    lat=np.asarray(lat,float)
    good=np.isfinite(lon)&np.isfinite(lat)
    lon,lat=lon[good],lat[good]

    if len(lon)<2:
        return lon,lat

    seg=haversine_km(
        lon[:-1],lat[:-1],
        lon[1:],lat[1:]
    )
    s=np.r_[0,np.cumsum(seg)]
    keep=np.r_[True,np.diff(s)>1e-6]
    lon,lat,s=lon[keep],lat[keep],s[keep]

    if len(lon)<2:
        return lon,lat

    ss=np.linspace(0,s[-1],n)
    return (
        np.interp(ss,s,lon),
        np.interp(ss,s,lat)
    )

def temporal_initial_path(dt,n=220):
    pd_prev,prev=accepted_neighbor(dt,"prev")
    pd_next,nxt=accepted_neighbor(dt,"next")

    if prev is None and nxt is None:
        return None

    if prev is not None:
        p_lon,p_lat=resample_polyline(
            prev["lon"],prev["lat"],n=n
        )
    if nxt is not None:
        n_lon,n_lat=resample_polyline(
            nxt["lon"],nxt["lat"],n=n
        )

    if prev is not None and nxt is not None:
        den=max(1,(pd_next-pd_prev).days)
        alpha=np.clip(
            (pd.Timestamp(dt)-pd_prev).days/den,
            0,1
        )
        return (
            (1-alpha)*p_lon+alpha*n_lon,
            (1-alpha)*p_lat+alpha*n_lat,
            pd_prev,pd_next
        )

    if prev is not None:
        return p_lon,p_lat,pd_prev,None
    return n_lon,n_lat,None,pd_next

def make_speed_interp(field):
    speed=np.hypot(
        field["u"],field["v"]
    ).astype(np.float32)
    itp=RegularGridInterpolator(
        (field["lat"],field["lon"]),
        speed,
        bounds_error=False,
        fill_value=np.nan
    )
    return speed,itp

def interp_speed(itp,lon,lat):
    pts=np.column_stack([
        np.asarray(lat).ravel(),
        np.asarray(lon).ravel()
    ])
    return itp(pts).reshape(
        np.asarray(lon).shape
    )

def snap_control_point(
    lon0,lat0,field,speed_itp
):
    lat_scale=111.32
    lon_scale=111.32*np.cos(
        np.deg2rad(lat0)
    )

    dl=cfg.CONTROL_SNAP_RADIUS_KM/max(lon_scale,1e-6)
    da=cfg.CONTROL_SNAP_RADIUS_KM/lat_scale

    ix=np.where(
        (field["lon"]>=lon0-dl)&
        (field["lon"]<=lon0+dl)
    )[0]
    iy=np.where(
        (field["lat"]>=lat0-da)&
        (field["lat"]<=lat0+da)
    )[0]

    if not len(ix) or not len(iy):
        return float(lon0),float(lat0)

    LLON,LLAT=np.meshgrid(
        field["lon"][ix],
        field["lat"][iy]
    )
    sp=np.hypot(
        field["u"][np.ix_(iy,ix)],
        field["v"][np.ix_(iy,ix)]
    ).astype(float)

    dist=haversine_km(
        lon0,lat0,LLON,LLAT
    )
    mask=(
        np.isfinite(sp)&
        (dist<=cfg.CONTROL_SNAP_RADIUS_KM)
    )

    if not mask.any():
        return float(lon0),float(lat0)

    smax=np.nanmax(sp[mask])
    score=(
        sp/(smax+1e-8)-
        cfg.CONTROL_SNAP_DISTANCE_PENALTY*
        dist/cfg.CONTROL_SNAP_RADIUS_KM
    )
    score[~mask]=-np.inf

    j,i=np.unravel_index(
        np.nanargmax(score),score.shape
    )
    return float(LLON[j,i]),float(LLAT[j,i])

def smooth_control_polyline(points):
    pts=np.asarray(points,float)
    lon,lat=pts[:,0],pts[:,1]

    seg=haversine_km(
        lon[:-1],lat[:-1],
        lon[1:],lat[1:]
    )
    s=np.r_[0,np.cumsum(seg)]
    keep=np.r_[True,np.diff(s)>1e-6]
    lon,lat,s=lon[keep],lat[keep],s[keep]

    n=max(
        60,
        int(np.ceil(
            s[-1]/cfg.PATH_SAMPLE_SPACING_KM
        ))+1
    )
    ss=np.linspace(0,s[-1],n)

    if len(lon)>=3:
        fx=PchipInterpolator(s,lon)
        fy=PchipInterpolator(s,lat)
        return fx(ss),fy(ss)

    return (
        np.interp(ss,s,lon),
        np.interp(ss,s,lat)
    )

def ridge_refine(lon,lat,speed_itp):
    lon=np.asarray(lon,float).copy()
    lat=np.asarray(lat,float).copy()

    offsets=np.linspace(
        -cfg.RIDGE_HALF_KM,
        cfg.RIDGE_HALF_KM,
        cfg.RIDGE_SAMPLES
    )

    for _ in range(cfg.RIDGE_REFINE_PASSES):
        new_lon,new_lat=lon.copy(),lat.copy()

        for i in range(1,len(lon)-1):
            tangent=angle_east_rad(
                lon[i-1],lat[i-1],
                lon[i+1],lat[i+1]
            )

            xs=[];ys=[]
            for off in offsets:
                if abs(off)<1e-9:
                    x,y=lon[i],lat[i]
                else:
                    b=(
                        tangent+
                        (np.pi/2 if off>0 else -np.pi/2)
                    )
                    x,y=forward_point(
                        lon[i],lat[i],
                        b,abs(float(off))
                    )
                xs.append(x);ys.append(y)

            xs=np.asarray(xs)
            ys=np.asarray(ys)
            sp=interp_speed(
                speed_itp,xs,ys
            )

            good=np.isfinite(sp)
            if not good.any():
                continue

            smax=np.nanmax(sp[good])
            score=(
                sp/(smax+1e-8)-
                cfg.RIDGE_DISTANCE_PENALTY*
                np.abs(offsets)/cfg.RIDGE_HALF_KM
            )
            score[~good]=-np.inf

            k=int(np.nanargmax(score))
            new_lon[i]=xs[k]
            new_lat[i]=ys[k]

        lon,lat=new_lon,new_lat

    return lon,lat

def build_manual_path(points,field):
    speed,itp=make_speed_interp(field)

    snapped=[
        snap_control_point(
            x,y,field,itp
        )
        for x,y in points
    ]

    lon,lat=smooth_control_polyline(
        snapped
    )
    lon,lat=ridge_refine(
        lon,lat,itp
    )

    return {
        "lon":np.asarray(lon,np.float32),
        "lat":np.asarray(lat,np.float32),
        "control_points_raw":np.asarray(points,np.float32),
        "control_points_snapped":np.asarray(snapped,np.float32),
    }

def diagnostics(path,field):
    lon=np.asarray(path["lon"],float)
    lat=np.asarray(path["lat"],float)
    speed,itp=make_speed_interp(field)

    sp=interp_speed(itp,lon,lat)
    finite=float(
        np.isfinite(sp).mean()
    )

    rr=[]
    for i in range(2,len(lon)-2,2):
        tangent=angle_east_rad(
            lon[i-1],lat[i-1],
            lon[i+1],lat[i+1]
        )
        offs=np.linspace(-35,35,15)
        xs=[];ys=[]
        for off in offs:
            if abs(off)<1e-9:
                x,y=lon[i],lat[i]
            else:
                b=tangent+(
                    np.pi/2 if off>0 else -np.pi/2
                )
                x,y=forward_point(
                    lon[i],lat[i],b,
                    abs(float(off))
                )
            xs.append(x);ys.append(y)

        ss=interp_speed(
            itp,np.asarray(xs),np.asarray(ys)
        )

        if np.isfinite(ss).any() and np.isfinite(sp[i]):
            rr.append(
                float(
                    sp[i]/
                    (np.nanmax(ss)+1e-8)
                )
            )

    rr=np.asarray(rr,float)

    if len(lon)>3:
        dirs=np.array([
            angle_east_rad(
                lon[i],lat[i],
                lon[i+1],lat[i+1]
            )
            for i in range(len(lon)-1)
        ])
        turns=np.abs(
            np.rad2deg(
                wrap_pi(np.diff(dirs))
            )
        )
        p95=float(
            np.nanpercentile(turns,95)
        )
    else:
        p95=np.nan

    return {
        "max_lon":float(np.nanmax(lon)),
        "finite_speed_frac":finite,
        "ridge_median":float(np.nanmedian(rr)) if len(rr) else np.nan,
        "ridge_p10":float(np.nanpercentile(rr,10)) if len(rr) else np.nan,
        "p95_turn_deg":p95,
    }


In [ ]:

# ===== 3. 交互人工复核器 =====
class LongManualQC:
    def __init__(self,dates):
        self.dates=dates
        self.index=0
        self.dt=None
        self.field=None
        self.speed=None
        self.control_points=[]
        self.path=None
        self.metrics=None

        self.fig,self.ax=plt.subplots(
            figsize=(11.5,7.5)
        )
        self.fig.canvas.mpl_connect(
            "button_press_event",
            self.on_click
        )

        self.title=widgets.HTML()
        self.status=widgets.HTML()

        self.prev_btn=widgets.Button(description="◀")
        self.next_btn=widgets.Button(description="▶")
        self.accept_btn=widgets.Button(
            description="批准自动路径",
            button_style="success"
        )
        self.init_btn=widgets.Button(
            description="邻日初始化"
        )
        self.refine_btn=widgets.Button(
            description="构建/细化",
            button_style="info"
        )
        self.undo_btn=widgets.Button(
            description="撤销"
        )
        self.clear_btn=widgets.Button(
            description="清空"
        )
        self.save_btn=widgets.Button(
            description="保存人工路径",
            button_style="success"
        )
        self.unusable_btn=widgets.Button(
            description="标记不可用",
            button_style="warning"
        )

        self.quiver=widgets.Checkbox(
            value=False,
            description="显示流向箭头"
        )
        self.confirm=widgets.Checkbox(
            value=False,
            description="我已确认路径沿当天黑潮主流轴"
        )
        self.note=widgets.Textarea(
            description="备注",
            layout=widgets.Layout(
                width="760px",
                height="65px"
            )
        )

        self.prev_btn.on_click(
            lambda b:self.move(-1)
        )
        self.next_btn.on_click(
            lambda b:self.move(1)
        )
        self.accept_btn.on_click(
            self.accept_auto
        )
        self.init_btn.on_click(
            self.auto_init
        )
        self.refine_btn.on_click(
            self.refine
        )
        self.undo_btn.on_click(
            self.undo
        )
        self.clear_btn.on_click(
            self.clear
        )
        self.save_btn.on_click(
            self.save_manual
        )
        self.unusable_btn.on_click(
            self.mark_unusable
        )
        self.quiver.observe(
            lambda ch:self.redraw(),
            names="value"
        )

        box=widgets.VBox([
            self.title,
            widgets.HBox([
                self.prev_btn,self.next_btn,
                self.accept_btn,self.init_btn,
                self.refine_btn,self.undo_btn,
                self.clear_btn,self.save_btn,
                self.unusable_btn
            ]),
            widgets.HBox([
                self.quiver,self.confirm
            ]),
            self.note,
            self.status
        ])

        display(box)
        display(self.fig.canvas)
        self.load_date(0)

    def meta_path(self,dt):
        return MANUAL_META/f"{dt:%Y%m%d}.json"

    def manual_track_path(self,dt):
        return MANUAL_TRACK/f"{dt:%Y%m%d}.npz"

    def set_status(self,msg,color="#333"):
        self.status.value=(
            f"<div style='color:{color}'>"
            f"{msg}</div>"
        )

    def load_date(self,index):
        self.index=int(
            np.clip(index,0,len(self.dates)-1)
        )
        self.dt=self.dates[self.index]
        self.field=load_field(self.dt)
        self.speed=np.hypot(
            self.field["u"],
            self.field["v"]
        )
        self.control_points=[]
        self.path=None
        self.metrics=None
        self.confirm.value=False
        self.note.value=""

        mp=self.meta_path(self.dt)
        if mp.exists():
            md=json.loads(
                mp.read_text(encoding="utf-8")
            )
            self.note.value=md.get("note","")

        reason=queue[
            queue.date==self.dt
        ].iloc[0].get("reasons","")

        done=sum(
            self.meta_path(d).exists()
            for d in self.dates
        )

        self.title.value=(
            f"<h3>{self.index+1}/{len(self.dates)} "
            f"{self.dt.date()}</h3>"
            f"<b>原因:</b> {reason} "
            f"&nbsp;&nbsp;<b>已处理:</b> "
            f"{done}/{len(self.dates)}"
        )

        self.redraw()

    def move(self,d):
        self.load_date(
            self.index+d
        )

    def redraw(self):
        self.ax.clear()

        st=max(1,cfg.DISPLAY_STRIDE)
        self.ax.imshow(
            self.speed[::st,::st],
            origin="lower",
            extent=[
                self.field["lon"][0],
                self.field["lon"][-1],
                self.field["lat"][0],
                self.field["lat"][-1],
            ],
            aspect="auto",
            interpolation="nearest"
        )

        a=auto_track(self.dt)
        if a is not None:
            self.ax.plot(
                a["lon"],a["lat"],
                lw=1.4,ls="--",
                label="auto track"
            )

        pd_prev,prev=accepted_neighbor(
            self.dt,"prev"
        )
        pd_next,nxt=accepted_neighbor(
            self.dt,"next"
        )

        if prev is not None:
            self.ax.plot(
                prev["lon"],prev["lat"],
                color="0.60",ls="--",
                lw=1.0,
                label=f"prev {pd_prev.date()}"
            )
        if nxt is not None:
            self.ax.plot(
                nxt["lon"],nxt["lat"],
                color="0.35",ls=":",
                lw=1.0,
                label=f"next {pd_next.date()}"
            )

        leg=legacy_track(self.dt)
        if leg is not None:
            self.ax.plot(
                leg["lon"],leg["lat"],
                color="purple",
                ls="-.",
                lw=1.0,
                alpha=.75,
                label="legacy 2023–25 reference"
            )

        if self.quiver.value:
            qs=max(4,cfg.QUIVER_STRIDE)
            self.ax.quiver(
                self.field["lon"][::qs],
                self.field["lat"][::qs],
                self.field["u"][::qs,::qs],
                self.field["v"][::qs,::qs],
                color="white",
                alpha=.55,
                scale=25,
                width=.0018
            )

        if self.control_points:
            p=np.asarray(
                self.control_points,float
            )
            self.ax.plot(
                p[:,0],p[:,1],
                "o-",ms=4,lw=.8,
                label="manual control"
            )

        if self.path is not None:
            self.ax.plot(
                self.path["lon"],
                self.path["lat"],
                lw=2.2,
                label="manual refined"
            )

        self.ax.set_xlim(
            cfg.XMIN,cfg.XMAX
        )
        self.ax.set_ylim(
            cfg.YMIN,cfg.YMAX
        )
        self.ax.set_xlabel("Longitude")
        self.ax.set_ylabel("Latitude")
        self.ax.set_title(
            f"{self.dt.date()} | 50 m speed"
        )
        self.ax.legend(
            loc="best",fontsize=7
        )
        self.fig.canvas.draw_idle()

    def on_click(self,event):
        if (
            event.inaxes!=self.ax or
            event.xdata is None
        ):
            return

        if event.button==1:
            self.control_points.append(
                (
                    float(event.xdata),
                    float(event.ydata)
                )
            )
            self.path=None
            self.confirm.value=False
            self.redraw()

        elif (
            event.button==3 and
            self.control_points
        ):
            x,y=event.xdata,event.ydata
            d=[
                float(haversine_km(
                    px,py,x,y
                ))
                for px,py in self.control_points
            ]
            self.control_points.pop(
                int(np.argmin(d))
            )
            self.path=None
            self.confirm.value=False
            self.redraw()

    def auto_init(self,_):
        init=temporal_initial_path(
            self.dt
        )
        if init is None:
            self.set_status(
                "无邻日初始化路径","#b00"
            )
            return

        lon,lat,pd_prev,pd_next=init
        ids=np.linspace(
            0,len(lon)-1,
            cfg.TEMPORAL_INIT_CONTROL_POINTS
        ).astype(int)

        self.control_points=[
            (float(lon[i]),float(lat[i]))
            for i in ids
        ]
        self.refine(None)
        self.set_status(
            "已用邻日初始化；必须根据当天速度场人工确认。",
            "#8a5200"
        )

    def refine(self,_):
        if len(self.control_points)<cfg.MIN_CONTROL_POINTS:
            self.set_status(
                f"控制点不足，至少 {cfg.MIN_CONTROL_POINTS} 个",
                "#b00"
            )
            return

        self.path=build_manual_path(
            self.control_points,
            self.field
        )
        self.metrics=diagnostics(
            self.path,self.field
        )
        self.confirm.value=False
        self.redraw()

        m=self.metrics
        warn=[]
        if m["max_lon"]<cfg.MIN_MAX_LON:
            warn.append("未到146E")
        if m["finite_speed_frac"]<cfg.MIN_FINITE_SPEED_FRAC:
            warn.append("无效流速点偏多")
        if (
            np.isfinite(m["ridge_median"]) and
            m["ridge_median"]<cfg.WARN_RIDGE_MEDIAN
        ):
            warn.append("ridge median偏低")
        if (
            np.isfinite(m["ridge_p10"]) and
            m["ridge_p10"]<cfg.WARN_RIDGE_P10
        ):
            warn.append("ridge P10偏低")
        if (
            np.isfinite(m["p95_turn_deg"]) and
            m["p95_turn_deg"]>cfg.WARN_P95_TURN_DEG
        ):
            warn.append("转向偏大")

        txt=(
            f"max_lon={m['max_lon']:.2f} | "
            f"finite={m['finite_speed_frac']:.1%} | "
            f"ridge_med={m['ridge_median']:.3f} | "
            f"ridge_p10={m['ridge_p10']:.3f} | "
            f"P95_turn={m['p95_turn_deg']:.1f}"
        )
        if warn:
            txt+=" | WARN: "+"; ".join(warn)
            self.set_status(
                txt,"#b05a00"
            )
        else:
            self.set_status(
                txt,"#087f23"
            )

    def undo(self,_):
        if self.control_points:
            self.control_points.pop()
            self.path=None
            self.confirm.value=False
            self.redraw()

    def clear(self,_):
        self.control_points=[]
        self.path=None
        self.metrics=None
        self.confirm.value=False
        self.redraw()

    def write_meta(self,status,extra=None):
        obj={
            "date":str(self.dt.date()),
            "status":status,
            "note":self.note.value,
        }
        if extra:
            obj.update(extra)

        self.meta_path(self.dt).write_text(
            json.dumps(
                obj,indent=2,
                ensure_ascii=False
            ),
            encoding="utf-8"
        )

    def accept_auto(self,_):
        a=auto_track(self.dt)
        if a is None or not a["completed"]:
            self.set_status(
                "自动路径未完成，不能直接批准。",
                "#b00"
            )
            return

        if not self.confirm.value:
            self.set_status(
                "先勾选“我已确认路径沿当天黑潮主流轴”。",
                "#b00"
            )
            return

        self.write_meta(
            "accepted_auto_after_manual_qc"
        )
        self.set_status(
            "已批准自动路径。",
            "#087f23"
        )
        if self.index<len(self.dates)-1:
            self.load_date(
                self.index+1
            )

    def save_manual(self,_):
        if self.path is None:
            self.set_status(
                "请先构建人工路径。",
                "#b00"
            )
            return

        if not self.confirm.value:
            self.set_status(
                "请先勾选人工确认。",
                "#b00"
            )
            return

        m=self.metrics
        if m["max_lon"]<cfg.MIN_MAX_LON:
            self.set_status(
                "人工路径未到145.8E，不能保存。",
                "#b00"
            )
            return

        np.savez_compressed(
            self.manual_track_path(self.dt),
            lon=self.path["lon"],
            lat=self.path["lat"],
            completed=np.array(True),
            stop_reason=np.array(
                "manual_qc_corrected"
            ),
            tracker_version=np.array(
                "LONG_1993_2025_MANUAL_V1"
            ),
            control_points_raw=self.path["control_points_raw"],
            control_points_snapped=self.path["control_points_snapped"],
        )

        self.write_meta(
            "manual_corrected",
            {
                "metrics":{
                    k:(
                        None if not np.isfinite(v)
                        else float(v)
                    )
                    for k,v in m.items()
                }
            }
        )

        self.fig.savefig(
            MANUAL_FIG/
            f"{self.dt:%Y%m%d}_manual_qc.png",
            dpi=110,
            bbox_inches="tight"
        )

        self.set_status(
            "人工路径已保存。",
            "#087f23"
        )

        if self.index<len(self.dates)-1:
            self.load_date(
                self.index+1
            )

    def mark_unusable(self,_):
        if not self.confirm.value:
            self.set_status(
                "若确定该日不可用，也请先勾选人工确认。",
                "#b00"
            )
            return

        self.write_meta(
            "unusable"
        )
        self.set_status(
            "已标记该日不可用。",
            "#8a5200"
        )
        if self.index<len(self.dates)-1:
            self.load_date(
                self.index+1
            )

annotator=LongManualQC(DATES)



## 复核原则

这一步是数据清理，不是独立验证。

判断顺序建议：

1. **当天 50 m speed field** 是主体；
2. 自动路径若沿连续高速主流轴，直接批准；
3. 如果自动路径明显陷入涡旋、走错强流支路或提前终止，再手工修改；
4. 前后日路径用于帮助区分“连续主流”与“瞬时涡旋”，但不能替代当天流场；
5. 2023–2025 旧人工结果只是参考，不自动覆盖新的 multiyear 结果。


In [ ]:

# ===== 4. 人工复核完成度 =====
rows=[]

for dt in DATES:
    mp=MANUAL_META/f"{dt:%Y%m%d}.json"
    status="pending"
    if mp.exists():
        try:
            status=json.loads(
                mp.read_text(encoding="utf-8")
            ).get("status","unknown")
        except Exception:
            status="broken_meta"

    rows.append({
        "date":dt,
        "status":status,
        "manual_track_exists":(
            MANUAL_TRACK/f"{dt:%Y%m%d}.npz"
        ).exists()
    })

manual_status=pd.DataFrame(rows)
manual_status.to_csv(
    MANUAL_REPORT/
    "人工复核完成度.csv",
    index=False,
    encoding="utf-8-sig"
)

display(
    manual_status["status"]
    .value_counts()
    .to_frame("count")
)

pending=manual_status[
    manual_status.status=="pending"
]
print("pending:",len(pending))


In [ ]:

# ===== 5. 全部人工队列处理后，冻结最终tracks =====
manual_status=pd.read_csv(
    MANUAL_REPORT/
    "人工复核完成度.csv",
    parse_dates=["date"]
)

pending=manual_status[
    manual_status.status=="pending"
]

if len(pending):
    raise RuntimeError(
        "还有未处理人工日期: "+
        ", ".join(
            pending.date.dt.strftime("%Y-%m-%d").tolist()[:30]
        )
    )

# 清空最终tracks
for p in FINAL_TRACK.glob("*.npz"):
    p.unlink()

manual_map={
    pd.Timestamp(r.date):r.status
    for r in manual_status.itertuples(index=False)
}

final_rows=[]

for r in manifest.itertuples(index=False):
    dt=pd.Timestamp(r.date)

    if r.raw_status!="present":
        final_rows.append({
            "date":dt,
            "status":"missing_raw",
            "source":r.selected_source
        })
        continue

    auto_row=auto_report[
        auto_report.date==dt
    ]
    if not len(auto_row):
        final_rows.append({
            "date":dt,
            "status":"no_auto_result",
            "source":r.selected_source
        })
        continue

    is_manual=dt in manual_map

    if is_manual:
        st=manual_map[dt]

        if st=="unusable":
            final_rows.append({
                "date":dt,
                "status":"unusable",
                "source":r.selected_source
            })
            continue

        if st=="manual_corrected":
            src=MANUAL_TRACK/f"{dt:%Y%m%d}.npz"
            dst=FINAL_TRACK/src.name
            shutil.copy2(src,dst)

            final_rows.append({
                "date":dt,
                "status":"final",
                "source":"manual_corrected"
            })
            continue

        if st=="accepted_auto_after_manual_qc":
            src=AUTO_TRACK/f"{dt:%Y%m%d}.npz"
            dst=FINAL_TRACK/src.name
            shutil.copy2(src,dst)

            final_rows.append({
                "date":dt,
                "status":"final",
                "source":"auto_approved_manual_qc"
            })
            continue

        raise RuntimeError(
            f"{dt.date()} 未知人工状态 {st}"
        )

    # 非人工队列：必须是accepted_auto
    ar=auto_row.iloc[0]
    if ar.status=="accepted_auto":
        src=AUTO_TRACK/f"{dt:%Y%m%d}.npz"
        dst=FINAL_TRACK/src.name
        shutil.copy2(src,dst)

        final_rows.append({
            "date":dt,
            "status":"final",
            "source":"accepted_auto"
        })
    else:
        raise RuntimeError(
            f"{dt.date()} status={ar.status} "
            "但没有进入人工队列。"
        )

final_index=pd.DataFrame(
    final_rows
).sort_values("date")

final_index.to_csv(
    FINAL_REPORT/
    "最终逐日路径索引.csv",
    index=False,
    encoding="utf-8-sig"
)

print(
    final_index.status.value_counts()
)
print(
    "final tracks:",
    len(list(FINAL_TRACK.glob("*.npz")))
)


In [ ]:

# ===== 6. 构建1993–2025最终dual-branch dataset =====
TARGET_LATS_1=np.arange(
    22.0,25.5+1e-9,1/12
)
TARGET_LONS_2=np.arange(
    122.0,145.0+1e-9,1/12
)

def branch1_from_track(lons,lats):
    valid=(
        np.isfinite(lons)&
        np.isfinite(lats)&
        (lats<=25.7)
    )
    lons=np.asarray(lons)[valid]
    lats=np.asarray(lats)[valid]

    if len(lats)<2:
        return None

    df=(
        pd.DataFrame({
            "lat":lats,
            "lon":lons
        })
        .groupby("lat",as_index=False)
        .lon.mean()
        .sort_values("lat")
    )

    if len(df)<2:
        return None

    f=interp1d(
        df.lat.values,
        df.lon.values,
        bounds_error=False,
        fill_value="extrapolate"
    )
    return f(
        TARGET_LATS_1
    ).astype(np.float32)

def branch2_from_track(lons,lats):
    valid=(
        np.isfinite(lons)&
        np.isfinite(lats)
    )
    lons=np.asarray(lons)[valid]
    lats=np.asarray(lats)[valid]

    if len(lons)<2:
        return None

    vals=[]
    for lo in np.sort(
        np.unique(lons)
    ):
        yy=lats[lons==lo]
        vals.append((
            lo,
            np.max(yy)
            if lo<=125
            else np.mean(yy)
        ))

    arr=np.asarray(vals,float)

    if len(arr)<2:
        return None

    f=interp1d(
        arr[:,0],arr[:,1],
        bounds_error=False,
        fill_value="extrapolate"
    )
    return f(
        TARGET_LONS_2
    ).astype(np.float32)

D=[];B1=[];B2=[];SRC=[]

index_map={
    pd.Timestamp(r.date):r.source
    for r in final_index[
        final_index.status=="final"
    ].itertuples(index=False)
}

for p in tqdm(
    sorted(FINAL_TRACK.glob("*.npz")),
    desc="dual branch"
):
    dt=pd.Timestamp(p.stem)

    with np.load(
        p,allow_pickle=False
    ) as z:
        lon=np.asarray(z["lon"],float)
        lat=np.asarray(z["lat"],float)

    b1=branch1_from_track(
        lon,lat
    )
    b2=branch2_from_track(
        lon,lat
    )

    if (
        b1 is not None and
        b2 is not None and
        np.isfinite(b1).all() and
        np.isfinite(b2).all()
    ):
        D.append(dt)
        B1.append(b1)
        B2.append(b2)
        SRC.append(
            index_map.get(dt,"unknown")
        )

B1=np.stack(B1)
B2=np.stack(B2)

dataset_path=(
    FINAL_DATASET/
    "daily_dual_branch_paths_1993_2025_FINAL.npz"
)

np.savez_compressed(
    dataset_path,
    dates=np.array([
        d.strftime("%Y-%m-%d")
        for d in D
    ]),
    branch1_lon=B1,
    branch1_target_lats=TARGET_LATS_1,
    branch2_lat=B2,
    branch2_target_lons=TARGET_LONS_2,
    source=np.asarray(SRC)
)

meta=pd.DataFrame({
    "date":D,
    "source":SRC
})
meta.to_csv(
    FINAL_DATASET/
    "daily_dual_branch_meta_1993_2025_FINAL.csv",
    index=False,
    encoding="utf-8-sig"
)

print("dataset:",dataset_path)
print("final dual-branch days:",len(D))
print(
    "range:",
    min(D).date(),"->",max(D).date()
)
display(
    meta.source.value_counts()
)


In [ ]:

# ===== 7. 最终覆盖率 / 连续区间 / handoff =====
Didx=pd.DatetimeIndex(D).sort_values()
calendar=pd.date_range(
    Didx.min(),
    Didx.max(),
    freq="D"
)
available=pd.Series(
    False,index=calendar
)
available.loc[Didx]=True

# 连续有效run
runs=[]
start=None

for dt,val in available.items():
    if val and start is None:
        start=dt
    if (
        start is not None and
        (
            (not val) or
            dt==available.index[-1]
        )
    ):
        end=(
            dt
            if val and dt==available.index[-1]
            else dt-pd.Timedelta(days=1)
        )
        runs.append({
            "start":start,
            "end":end,
            "days":(end-start).days+1
        })
        start=None

runs_df=pd.DataFrame(runs).sort_values(
    "days",ascending=False
)

runs_df.to_csv(
    FINAL_REPORT/
    "最终连续有效时间段.csv",
    index=False,
    encoding="utf-8-sig"
)

annual=(
    meta.assign(year=meta.date.dt.year)
    .groupby("year",as_index=False)
    .agg(valid_days=("date","size"))
)

annual.to_csv(
    FINAL_REPORT/
    "最终年度有效路径统计.csv",
    index=False,
    encoding="utf-8-sig"
)

handoff={
    "dataset":str(dataset_path),
    "date_start":str(Didx.min().date()),
    "date_end":str(Didx.max().date()),
    "final_daily_paths":int(len(Didx)),
    "calendar_days":int(len(calendar)),
    "coverage":float(len(Didx)/len(calendar)),
    "manual_queue_total":int(len(DATES)),
    "manual_status_counts":(
        manual_status.status
        .value_counts()
        .to_dict()
    ),
    "longest_continuous_runs":(
        runs_df.head(10)
        .astype(str)
        .to_dict(orient="records")
    ),
}

(FINAL_REPORT/
 "LONG_DAILY_1993_2025_HANDOFF.json").write_text(
    json.dumps(
        handoff,indent=2,
        ensure_ascii=False
    ),
    encoding="utf-8"
)

print(json.dumps(
    handoff,indent=2,
    ensure_ascii=False
))
